# 03 — State B FEA Prompt Generation

Build the final FEA prompt from State A and show the exact prompt text.


In [ ]:
from pathlib import Path
import json
import logging
import os
import sys

import yaml
from IPython.display import Image, Markdown, display

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "code_base").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
MODULE_ROOT = PROJECT_ROOT / "code_base" / "fea_cad_one_sample"
if str(MODULE_ROOT) not in sys.path:
    sys.path.insert(0, str(MODULE_ROOT))

API_ENV_PATH = MODULE_ROOT / "src" / "api.env"

def _load_api_env(path: Path) -> dict[str, str]:
    env_values: dict[str, str] = {}
    if not path.exists():
        return env_values
    for raw_line in path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        key = key.strip()
        value = value.strip().strip("'").strip('"')
        if key and value:
            env_values[key] = value
            os.environ.setdefault(key, value)
    return env_values

api_env_values = _load_api_env(API_ENV_PATH)

from src import interfaces as api

logging.basicConfig(level=logging.INFO, format="%(name)s | %(levelname)s | %(message)s")

FIXED_SAMPLE_PATH = MODULE_ROOT / "experiment_config" / "fixed_sample.yaml"
OUTPUT_ROOT = MODULE_ROOT / "outputs"
sample_config = yaml.safe_load(FIXED_SAMPLE_PATH.read_text(encoding="utf-8"))
sample_id = str(sample_config["sample_id"])
selection_source = str(sample_config.get("load_source", "dataset"))
connection_string = os.environ.get("CAD_DB_CONNECTION_STRING")
state_a_dir = OUTPUT_ROOT / f"sample_{sample_id}" / "01_dataset_original"
state_b_dir = OUTPUT_ROOT / f"sample_{sample_id}" / "02_fea_constrained_revision"
state_b_dir.mkdir(parents=True, exist_ok=True)

print("[STAGE] Setup")
print(f"  → sample id      : {sample_id}")
print(f"  → selection mode : {selection_source}")
print(f"  → state A dir    : {state_a_dir}")
print(f"  → state B dir    : {state_b_dir}")
print(f"  → api.env        : {API_ENV_PATH}")
print(f"  → api.env keys   : {sorted(api_env_values)}")

In [ ]:
print("[STAGE] Load State A")
sample = api.load_selected_sample(
    module_root=MODULE_ROOT,
    sample_id=sample_id,
    selection_source=selection_source,
    connection_string=connection_string,
)
original_prompt_path = state_a_dir / "original_prompt.txt"
original_code_path = state_a_dir / "database_original_code.py"
state_a_views_dir = state_a_dir / "views"
assert original_prompt_path.exists()
assert original_code_path.exists()

original_prompt = original_prompt_path.read_text(encoding="utf-8")
original_code = original_code_path.read_text(encoding="utf-8")

display(Markdown(f"## Original {selection_source} prompt"))
display(Markdown("```text\n" + original_prompt + "\n```"))
display(Markdown("## State A views"))
for name in ["front", "side", "top", "iso", "grid"]:
    display(Markdown(f"**{name}**"))
    display(Image(filename=str(state_a_views_dir / f"{name}.png")))
print(f"  → code length : {len(original_code)}")
print(f"  → prompt len  : {len(original_prompt)}")
print(f"  → source      : {sample.source}")

In [ ]:
print("[STAGE] Build final FEA prompt")
load_case_path = state_b_dir / "load_case.json"
selector_hints_path = state_b_dir / "selector_hints.json"
prompt_path = state_b_dir / "fea_revision_prompt.txt"

load_case = api.write_load_case(sample.sample_id, load_case_path, force=True)
selector_hints = api.write_selector_hints(load_case, selector_hints_path, force=True)
fea_prompt = api.build_fea_prompt(original_prompt, original_code, load_case, selector_hints)
prompt_path.write_text(fea_prompt, encoding="utf-8")

print(f"  → prompt file   : {prompt_path}")
print(f"  → load case     : {load_case_path}")
print(f"  → selector hints: {selector_hints_path}")
print(f"  → prompt lines  : {len(fea_prompt.splitlines())}")
print(f"  → material      : {load_case.material}")
print(f"  → fixed region  : {selector_hints.fixed_region_description}")
print(f"  → load region   : {selector_hints.load_region_description}")


In [ ]:
display(Markdown("## Final FEA prompt"))
display(Markdown(f"```text\n{fea_prompt}\n```"))

assert prompt_path.exists()
assert "State A original prompt:" in fea_prompt
assert "State A original DB code:" in fea_prompt
assert "Load case (JSON):" in fea_prompt
assert "Selector hints (JSON):" in fea_prompt
assert "Preserve the original design identity." in fea_prompt
assert "Return only JSON containing code_lines and change_log." in fea_prompt


## What this notebook proves

- The DB-loaded State A sample is clear.
- The final FEA prompt is built and visible.
- The prompt file is saved for later notebooks.
